# Predict Customer Churn — Kaggle Playground Series S6E3

This documented notebook reproduces the **original submitted modeling workflow**: feature engineering, frequency/target encoding, nested stratified cross-validation, first-stage predictions from LightGBM, CatBoost and logistic regression, and a second-stage XGBoost classifier.

**Result recorded by the original run:** stacked out-of-fold ROC-AUC **0.916630** (five outer folds). The result is not a verified independent test score. See the [README](README.md) for methodological caveats, especially the use of outer validation labels for early stopping.

Competition: https://www.kaggle.com/competitions/playground-series-s6e3/overview

**To run:** attach the Playground Series S6E3 competition dataset to a Kaggle notebook. Input paths below match the original Kaggle environment. No dataset files are included in this repository.


## 1. Imports

Packages used by the executed modeling pipeline. Unused imports from the exploratory notebook have been removed; the modeling logic is unchanged.


In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


## 2. Data

The supervised target is `Churn` (Yes = 1, No = 0). The original notebook also loaded the public IBM Telco dataset into an `original` variable but **never used it in feature construction or model fitting**; that unused read is omitted here to avoid an unnecessary dataset dependency.


In [ ]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/test.csv")


## 3. Feature schema and initial cleaning

Categorical fields describe demographics, services, contract and payment choices. Numeric fields contain charges, tenure and senior-citizen status. The Kaggle `id` is removed from model features.


In [4]:
features_cat = ['gender', 'Partner', 'Dependents',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
features_num = ['MonthlyCharges', 'TotalCharges', 'tenure', 'SeniorCitizen']

In [5]:
train.drop(columns = {'id'}, inplace = True)
train['Churn'] = train['Churn'].map({"Yes": 1, "No": 0})

In [6]:
test.drop(columns = {'id'}, inplace = True)

## 4. Domain-inspired feature engineering

`engineer_telco_features` converts service and payment choices to numerical indicators; counts optional services; creates contract/tenure and price interactions; creates customer segments such as short tenure and month-to-month plans; and makes quantile bins.

**Note:** numeric medians and `qcut` bin boundaries are independently calculated when the function is called on train and test before CV, rather than being fitted on each training fold and reused. A production pipeline should fit those transformations on fold-training data only.


In [7]:
def engineer_telco_features(df):
    df = df.copy()

    # -----------------------------
    # Clean basic columns
    # -----------------------------
    cat_cols = [
        'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
        'InternetService', 'OnlineSecurity', 'OnlineBackup',
        'DeviceProtection', 'TechSupport', 'StreamingTV',
        'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod'
    ]
    for col in cat_cols:
        df[col] = df[col].fillna("Missing").astype(str)

    num_cols = ['MonthlyCharges', 'TotalCharges', 'tenure', 'SeniorCitizen']
    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # -----------------------------
    # Binary yes/no features
    # -----------------------------
    binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
    binary_map = {'Yes': 1, 'No': 0}
    for col in binary_cols:
        df[col] = df[col].map(binary_map)

    # gender
    df['gender'] = df['gender'].map({'Female': 0, 'Male': 1})

    # -----------------------------
    # Internet-related service columns
    # -----------------------------
    service_cols = [
        'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
        'TechSupport', 'StreamingTV', 'StreamingMovies'
    ]
    for col in service_cols:
        df[col] = df[col].replace({
            'No internet service': 0,
            'No': 0,
            'Yes': 1
        })

    # Count number of internet services
    df['ServiceCount'] = df[service_cols].sum(axis=1)

    # -----------------------------
    # MultipleLines
    # -----------------------------
    df['MultipleLines'] = df['MultipleLines'].replace({
        'No phone service': 0,
        'No': 0,
        'Yes': 1
    })

    # -----------------------------
    # InternetService
    # -----------------------------
    df['InternetService'] = df['InternetService'].map({
        'No': 0,
        'DSL': 1,
        'Fiber optic': 2
    })

    df['HasInternet'] = (df['InternetService'] > 0).astype(int)
    df['IsFiber'] = (df['InternetService'] == 2).astype(int)

    # -----------------------------
    # Contract
    # -----------------------------
    contract_map = {
        'Month-to-month': 1,
        'One year': 2,
        'Two year': 3
    }
    df['ContractNum'] = df['Contract'].map(contract_map)

    # -----------------------------
    # PaymentMethod
    # -----------------------------
    payment_map = {
        'Electronic check': 0,
        'Mailed check': 1,
        'Bank transfer (automatic)': 2,
        'Credit card (automatic)': 3
    }
    df['PaymentMethodNum'] = df['PaymentMethod'].map(payment_map)

    df['IsAutoPay'] = df['PaymentMethod'].isin([
        'Bank transfer (automatic)',
        'Credit card (automatic)'
    ]).astype(int)

    df['IsElectronicCheck'] = (df['PaymentMethod'] == 'Electronic check').astype(int)

    # -----------------------------
    # Impute numeric columns
    # -----------------------------
    for col in num_cols:
        med = df[col].median()
        df[col] = df[col].fillna(med)

    # -----------------------------
    # Interactions / ratios
    # -----------------------------
    df['ContractTenureScore'] = df['ContractNum'] * df['tenure']
    df['ChargesPerMonth'] = df['TotalCharges'] / (df['tenure'] + 1)
    df['MonthlyCharges_x_ServiceCount'] = df['MonthlyCharges'] * df['ServiceCount']

    # -----------------------------
    # Family
    # -----------------------------
    df['HasFamily'] = ((df['Dependents'] == 1) | (df['Partner'] == 1)).astype(int)

    # -----------------------------
    # Binning
    # -----------------------------
    df["monthly_bin5"] = pd.qcut(df["MonthlyCharges"], 5, labels=False, duplicates="drop")
    df["monthly_bin10"] = pd.qcut(df["MonthlyCharges"], 10, labels=False, duplicates="drop")

    df["tenure_bin5"] = pd.qcut(df["tenure"], 5, labels=False, duplicates="drop")
    df["tenure_bin10"] = pd.qcut(df["tenure"], 10, labels=False, duplicates="drop")

    df["total_bin5"] = pd.qcut(df["TotalCharges"], 5, labels=False, duplicates="drop")

    df["monthly_round"] = (df["MonthlyCharges"] // 10).astype(int)
    df["tenure_round"] = (df["tenure"] // 6).astype(int)

    # Fill any remaining qcut NaNs
    bin_cols = ["monthly_bin5", "monthly_bin10", "tenure_bin5", "tenure_bin10", "total_bin5"]
    for col in bin_cols:
        df[col] = df[col].fillna(-1).astype(int)

    # -----------------------------
    # Internet / price flags
    # -----------------------------
    df["fiber_high_price"] = (
        (df["InternetService"] == 2) &
        (df["MonthlyCharges"] > 80)
    ).astype(int)

    # -----------------------------
    # Stability
    # -----------------------------
    df["stable_customer"] = (
        (df["tenure"] > 24) &
        (df["ContractNum"] != 1)
    ).astype(int)

    df["unstable_customer"] = (
        (df["tenure"] < 12) &
        (df["ContractNum"] == 1)
    ).astype(int)

    # -----------------------------
    # Risk flags
    # -----------------------------
    df["high_price_flag"] = (df["MonthlyCharges"] > 80).astype(int)
    df["low_price_flag"] = (df["MonthlyCharges"] < 30).astype(int)

    df["risk_customer"] = (
        (df["ContractNum"] == 1) &
        (df["MonthlyCharges"] > 70)
    ).astype(int)

    df["high_price_short_tenure"] = (
        (df["tenure"] < 12) &
        (df["MonthlyCharges"] > 80)
    ).astype(int)

    # Addons
    df["num_addons"] = df[service_cols].sum(axis=1)

    df["no_addons_high_price"] = (
        (df["num_addons"] == 0) &
        (df["MonthlyCharges"] > 70)
    ).astype(int)

    return df

## 5. Frequency encoding and target encoding

For a categorical feature `col`, frequency encoding measures its proportion in a training split. Target encoding estimates the mean churn rate within each category. The function fits mappings using the supplied training split and applies them to validation/test, using the training target mean for unseen categories.

**Caveat:** the encoded **training** rows use means derived from those same rows' labels. That can encourage overfitting. A stronger implementation would use OOF/cross-fitted target encoding for every row used to train base models.


In [8]:
def frequency_target_encoding(train_df, valid_df, test_df, y_train, col):
    ## Frequency encoding ##
    freq_map = train_df[col].value_counts(normalize=True)

    train_df.loc[:, f"freq_{col}"] = train_df[col].map(freq_map).astype(float)
    valid_df.loc[:, f"freq_{col}"] = valid_df[col].map(freq_map).astype(float)
    test_df.loc[:, f"freq_{col}"] = test_df[col].map(freq_map).astype(float)
    ## Target encoding ##
    global_mean = y_train.mean()
    target_map = y_train.groupby(train_df[col]).mean()
    train_df.loc[:, f"te_{col}"] = train_df[col].map(target_map).astype(float)
    valid_df.loc[:, f"te_{col}"] = valid_df[col].map(target_map).astype(float)
    test_df.loc[:, f"te_{col}"] = test_df[col].map(target_map).astype(float)
    
    train_df.loc[:, f"freq_{col}"] = train_df[f"freq_{col}"].fillna(0.0)
    valid_df.loc[:, f"freq_{col}"] = valid_df[f"freq_{col}"].fillna(0.0)
    test_df.loc[:, f"freq_{col}"] = test_df[f"freq_{col}"].fillna(0.0)

    train_df.loc[:, f"te_{col}"] = train_df[f"te_{col}"].fillna(global_mean)
    valid_df.loc[:, f"te_{col}"] = valid_df[f"te_{col}"].fillna(global_mean)
    test_df.loc[:, f"te_{col}"] = test_df[f"te_{col}"].fillna(global_mean)
    return train_df, valid_df, test_df

## 6. Categorical pair interactions

The nine pairs combine correlated categorical contexts, such as `Contract + PaymentMethod`, `InternetService + TechSupport`, and `Partner + Dependents`. The resulting string features are frequency- and target-encoded.


In [9]:
pair_cols = [
    ("gender", "Partner"),
    ("Partner", "Dependents"),
    ("InternetService", "OnlineSecurity"),
    ("InternetService", "TechSupport"),
    ("InternetService", "StreamingTV"),
    ("InternetService", "StreamingMovies"),
    ("Contract", "PaymentMethod"),
    ("Contract", "PaperlessBilling"),
    ("MultipleLines", "PhoneService"),
]

def add_pair_features(df, pair_cols):
    df = df.copy()
    for c1, c2 in pair_cols:
        new_col = f"{c1}__{c2}"
        df[new_col] = df[c1].astype(str) + "__" + df[c2].astype(str)
    return df

## 7. Select model inputs

The notebook uses 15 base categorical features, 9 categorical pairs, 4 numeric features and 26 engineered numeric features. After expanding each of the 24 categorical features into frequency- and target-encoded variables, the first-stage models receive **78 numerical input columns**.


In [10]:
engineered_num_features = [
    'ServiceCount', 'HasInternet', 'IsFiber', 'ContractNum',
    'ContractTenureScore', 'PaymentMethodNum', 'IsAutoPay',
    'ChargesPerMonth', 'MonthlyCharges_x_ServiceCount', 'HasFamily',
    'monthly_bin5', 'monthly_bin10', 'tenure_bin5', 'tenure_bin10',
    'total_bin5', 'monthly_round', 'tenure_round', 'fiber_high_price',
    'stable_customer', 'unstable_customer', 'high_price_flag',
    'low_price_flag', 'risk_customer', 'high_price_short_tenure',
    'num_addons', 'no_addons_high_price'
]

In [11]:
train_work = train.copy()
test_work = test.copy()

train_work["Churn"] = train_work["Churn"].replace({"Yes": 1, "No": 0})
train_work["Churn"] = pd.to_numeric(train_work["Churn"], errors="coerce").astype(int)

train_fe = engineer_telco_features(train_work)
test_fe = engineer_telco_features(test_work)

train_fe = add_pair_features(train_fe, pair_cols)
test_fe = add_pair_features(test_fe, pair_cols)

pair_feature_names = [f"{c1}__{c2}" for c1, c2 in pair_cols]
cat_features_for_encoding = features_cat + pair_feature_names

# Keep raw cat columns for encoding stage plus numeric/engineered columns
base_features = features_cat + pair_feature_names + features_num + engineered_num_features

X = train_fe[base_features].copy()
y = train_fe["Churn"].copy()
X_test = test_fe[base_features].copy()

## 8. Model configuration

First-stage models: LightGBM, CatBoost and L2-regularized logistic regression (scaled features). Second-stage meta-model: XGBoost. XGBoost and Ridge alternatives in the original notebook were commented out as first-stage models and are not counted as active base learners.


In [12]:
cb_params = {
    "iterations": 8000,
    "learning_rate": 0.01,
    "depth": 5,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "auto_class_weights": "Balanced",
    "bootstrap_type": "Bernoulli",
    "subsample": 0.9,
    "l2_leaf_reg": 12,
    "random_strength": 1.2,
    "boosting_type": "Ordered",
    # "task_type": "GPU",
    "devices": "0",
    "verbose": False
}

lgbm_params = {
    "n_estimators": 5000,
    "learning_rate": 0.01,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 30,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "objective": "binary",
    "metric": "auc",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1
}

xgb_params = {
    "n_estimators": 3000,
    "learning_rate": 0.02,
    "max_depth": 4,
    "min_child_weight": 3,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 2.0,
    "reg_alpha": 0.2,
    "gamma": 0.0,
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "random_state": 42,
    "n_jobs": -1,
    "tree_method": "hist"
}

lr_params = {
    "C": 1.0,
    "penalty": "l2",
    "solver": "liblinear",
    "max_iter": 2000,
    "random_state": 42
}

## 9. Five outer folds × five inner folds, then stacking

For each outer fold:

1. Create five stratified inner folds using only the outer-training split.
2. Obtain out-of-fold predictions from LightGBM, CatBoost and logistic regression for each row in that outer-training split.
3. Refit those three learners on the full outer-training split and predict on the outer validation and Kaggle test data.
4. Add the three model predictions as meta-features alongside the engineered features.
5. Train an XGBoost meta-model and calculate outer-fold ROC-AUC.
6. Average the XGBoost test predictions from the five outer folds.

**Validation limitation:** CatBoost and LightGBM use each outer validation set for early stopping/model selection. Consequently the resulting outer-fold AUC is not a fully untouched nested-CV estimate. This notebook keeps the original published run's algorithm and numbers rather than silently changing it.


In [13]:
INNER_FOLD = 5
OUTER_FOLD = 5


cv1 = StratifiedKFold(n_splits = OUTER_FOLD, shuffle = True, random_state = 42)

oof_xgb = np.zeros(len(train))
pred_xgb = np.zeros(len(test))
outer_auc = []

for i, (train_idx, val_idx) in enumerate(cv1.split(X, y)):

    print(f"\nOuter Fold {i+1}/{OUTER_FOLD}")
    ## Outer Fold ##
    X_train_outer = X.iloc[train_idx].copy()
    X_val_outer = X.iloc[val_idx].copy()
    y_train_outer = y.iloc[train_idx].copy()
    y_val_outer = y.iloc[val_idx].copy()
    X_test_outer = X_test.copy()
    
    
    cv2 = StratifiedKFold(n_splits = INNER_FOLD, shuffle = True, random_state = 42)
    lgbm_oof_outer = np.zeros(len(X_train_outer))
    xgb_oof_outer = np.zeros(len(X_train_outer))
    # ridge_oof_outer = np.zeros(len(X_train_outer))
    lr_oof_outer = np.zeros(len(X_train_outer))
    cat_oof_outer = np.zeros(len(X_train_outer))
    
    for j, (train_idx2, val_idx2) in enumerate(cv2.split(X_train_outer, y_train_outer)):
        print(f"\nInner Fold {j+1}/{INNER_FOLD}")
        X_train_inner = X_train_outer.iloc[train_idx2].copy()
        X_val_inner = X_train_outer.iloc[val_idx2].copy()
        y_train_inner = y_train_outer.iloc[train_idx2].copy()
        y_val_inner = y_train_outer.iloc[val_idx2].copy()

        X_dummy = X_val_inner.copy()
        
        for col in cat_features_for_encoding:
            X_train_inner, X_val_inner, X_dummy = frequency_target_encoding(
                X_train_inner, X_val_inner, X_dummy, y_train_inner, col
            )
        
        freq_cols = [f"freq_{col}" for col in cat_features_for_encoding]
        te_cols = [f"te_{col}" for col in cat_features_for_encoding]
        model_features = features_num + engineered_num_features + freq_cols + te_cols
        
        X_train_inner_model = X_train_inner[model_features].copy()
        X_val_inner_model = X_val_inner[model_features].copy()
        ## CatBoost - Inner ##
        cat_model = CatBoostClassifier(**cb_params, random_seed=42)
        cat_model.fit(X_train_inner_model, y_train_inner, eval_set = (X_val_inner_model, y_val_inner), use_best_model=True,
        verbose=False)
        cat_oof_outer[val_idx2] = cat_model.predict_proba(X_val_inner_model)[:, 1]
        ## LGBM ##
        lgbm_model = lgb.LGBMClassifier(**lgbm_params, random_seed = 42)
        lgbm_model.fit(X_train_inner_model, y_train_inner, eval_set = [(X_val_inner_model, y_val_inner)], eval_metric="auc",
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
        lgbm_oof_outer[val_idx2] = lgbm_model.predict_proba(X_val_inner_model)[:, 1]
        ## XGBoost ##
        # xgb_model = xgb.XGBClassifier(**xgb_params)
        # xgb_model.fit(
        #     X_train_inner_model,
        #     y_train_inner,
        #     eval_set=[(X_val_inner_model, y_val_inner)],
        #     verbose=False
        # )
        # xgb_oof_outer[val_idx2] = xgb_model.predict_proba(X_val_inner_model)[:, 1]
        ## Ridge ##
        # scaler = StandardScaler()

        # X_train_inner_ridge = scaler.fit_transform(X_train_inner_model)
        # X_val_inner_ridge = scaler.transform(X_val_inner_model)
        
        # ridge_model = Ridge(alpha=1.0, random_state=42)
        # ridge_model.fit(X_train_inner_ridge, y_train_inner)
        
        # ridge_pred_inner = ridge_model.predict(X_val_inner_ridge)
        # ridge_pred_inner = np.clip(ridge_pred_inner, 0, 1)
        
        # ridge_oof_outer[val_idx2] = ridge_pred_inner
        ## Logistic Regression ##
        scaler = StandardScaler()

        X_train_inner_lr = scaler.fit_transform(X_train_inner_model)
        X_val_inner_lr = scaler.transform(X_val_inner_model)
        
        lr_model = LogisticRegression(
            C=1.0,
            penalty="l2",
            solver="liblinear",
            max_iter=2000,
            random_state=42
        )
        
        lr_model.fit(X_train_inner_lr, y_train_inner)
        lr_oof_outer[val_idx2] = lr_model.predict_proba(X_val_inner_lr)[:, 1]
   
        

    # -----------------------------
    # Fit full outer-train LGBM
    # -----------------------------
    X_train_full = X_train_outer.copy()
    X_val_full = X_val_outer.copy()
    X_test_full = X_test_outer.copy()

    for col in cat_features_for_encoding:
        X_train_full, X_val_full, X_test_full = frequency_target_encoding(
            X_train_full, X_val_full, X_test_full, y_train_outer, col
        )

    freq_cols = [f"freq_{col}" for col in cat_features_for_encoding]
    te_cols = [f"te_{col}" for col in cat_features_for_encoding]
    model_features = features_num + engineered_num_features + freq_cols + te_cols

    X_train_full_model = X_train_full[model_features].copy()
    X_val_full_model = X_val_full[model_features].copy()
    X_test_full_model = X_test_full[model_features].copy()

    lgbm_full = lgb.LGBMClassifier(**lgbm_params, random_seed=42)

    lgbm_full.fit(
        X_train_full_model,
        y_train_outer,
        eval_set=[(X_val_full_model, y_val_outer)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
    )

    lgbm_val_outer = lgbm_full.predict_proba(X_val_full_model)[:, 1]
    lgbm_test_outer = lgbm_full.predict_proba(X_test_full_model)[:, 1]
       
    # ## XGBoost Outer ##
    # xgb_full = xgb.XGBClassifier(**xgb_params)
    # xgb_full.fit(
    #     X_train_full_model,
    #     y_train_outer,
    #     eval_set=[(X_val_full_model, y_val_outer)],
    #     verbose=False
    # )
    # xgb_val_outer = xgb_full.predict_proba(X_val_full_model)[:, 1]
    # xgb_test_outer = xgb_full.predict_proba(X_test_full_model)[:, 1]

    ## Ridge Outer ##
    # scaler_full = StandardScaler()
    
    # X_train_full_ridge = scaler_full.fit_transform(X_train_full_model)
    # X_val_full_ridge = scaler_full.transform(X_val_full_model)
    # X_test_full_ridge = scaler_full.transform(X_test_full_model)
    
    # ridge_full = Ridge(alpha=1.0, random_state=42)
    # ridge_full.fit(X_train_full_ridge, y_train_outer)
    
    # ridge_val_outer = np.clip(ridge_full.predict(X_val_full_ridge), 0, 1)
    # ridge_test_outer = np.clip(ridge_full.predict(X_test_full_ridge), 0, 1)
    ## Logistic Regression ##
    scaler_full = StandardScaler()

    X_train_full_lr = scaler_full.fit_transform(X_train_full_model)
    X_val_full_lr = scaler_full.transform(X_val_full_model)
    X_test_full_lr = scaler_full.transform(X_test_full_model)
    
    lr_full = LogisticRegression(
        C=1.0,
        penalty="l2",
        solver="liblinear",
        max_iter=2000,
        random_state=42
    )
    
    lr_full.fit(X_train_full_lr, y_train_outer)
    
    lr_val_outer = lr_full.predict_proba(X_val_full_lr)[:, 1]
    lr_test_outer = lr_full.predict_proba(X_test_full_lr)[:, 1]

    ## CatBoost Outer ##

    cat_model_full = CatBoostClassifier(**cb_params, random_seed=42)
    cat_model_full.fit(X_train_full_model, y_train_outer, eval_set = (X_val_full_model, y_val_outer), use_best_model=True,
        verbose=False)
    cat_val_outer = cat_model_full.predict_proba(X_val_full_model)[:, 1]
    cat_test_outer = cat_model_full.predict_proba(X_test_full_model)[:, 1]
    

    # -----------------------------
    # CatBoost meta-model
    # -----------------------------
    # X_train_cat = X_train_full_model.copy()
    # X_val_cat = X_val_full_model.copy()
    # X_test_cat = X_test_full_model.copy()


    # X_train_cat["lgbm_pred"] = lgbm_oof_outer
    # X_train_cat["xgb_pred"] = xgb_oof_outer
    # # X_train_cat["ridge_pred"] = ridge_oof_outer
    # X_train_cat["lr_pred"] = lr_oof_outer
    
    # X_val_cat["lgbm_pred"] = lgbm_val_outer
    # X_val_cat["xgb_pred"] = xgb_val_outer
    # # X_val_cat["ridge_pred"] = ridge_val_outer
    # X_val_cat["lr_pred"] = lr_val_outer
    
    # X_test_cat["lgbm_pred"] = lgbm_test_outer
    # X_test_cat["xgb_pred"] = xgb_test_outer
    # # X_test_cat["ridge_pred"] = ridge_test_outer
    # X_test_cat["lr_pred"] = lr_test_outer

    # cb_model = CatBoostClassifier(**cb_params, random_seed=42 + i)

    # cb_model.fit(
    #     X_train_cat,
    #     y_train_outer,
    #     eval_set=(X_val_cat, y_val_outer),
    #     use_best_model=True,
    #     verbose=False
    # )

    # val_cat = cb_model.predict_proba(X_val_cat)[:, 1]
    # test_cat = cb_model.predict_proba(X_test_cat)[:, 1]

    ## XGBoost Meta Model ##
    X_train_xgb = X_train_full_model.copy()
    X_val_xgb = X_val_full_model.copy()
    X_test_xgb = X_test_full_model.copy()
    
    X_train_xgb["lgbm_pred"] = lgbm_oof_outer
    X_train_xgb["cat_pred"] = cat_oof_outer
    X_train_xgb["lr_pred"] = lr_oof_outer
    
    X_val_xgb["lgbm_pred"] = lgbm_val_outer
    X_val_xgb["cat_pred"] = cat_val_outer
    X_val_xgb["lr_pred"] = lr_val_outer
    
    X_test_xgb["lgbm_pred"] = lgbm_test_outer
    X_test_xgb["cat_pred"] = cat_test_outer
    X_test_xgb["lr_pred"] = lr_test_outer
    
    xgb_full = xgb.XGBClassifier(**xgb_params)
    xgb_full.fit(
        X_train_xgb,
        y_train_outer,
        eval_set=[(X_val_xgb, y_val_outer)],
        verbose=False
    )
    
    val_xgb = xgb_full.predict_proba(X_val_xgb)[:, 1]
    test_xgb = xgb_full.predict_proba(X_test_xgb)[:, 1]
        
    
    fold_auc = roc_auc_score(y_val_outer, val_xgb)
    outer_auc.append(fold_auc)
    
    oof_xgb[val_idx] = val_xgb
    pred_xgb += test_xgb / OUTER_FOLD
    
    print(f"Outer Fold {i+1} AUC: {fold_auc:.6f}")

print(f"\nStacked OOF AUC: {roc_auc_score(y, oof_xgb):.6f}")
print(f"Mean Outer AUC: {np.mean(outer_auc):.6f}")
print(f"Std Outer AUC: {np.std(outer_auc):.6f}")



    
        
        
    
    




Outer Fold 1/5

Inner Fold 1/5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[3852]	valid_0's auc: 0.914878

Inner Fold 2/5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[3115]	valid_0's auc: 0.916288

Inner Fold 3/5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[4243]	valid_0's auc: 0.917115

Inner Fold 4/5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[3276]	valid_0's auc: 0.915756

Inner Fold 5/5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[3368]	valid_0's auc: 0.917937
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[3796]	valid_0's auc: 0.91616
Outer Fold 1 AUC: 0.916370

Outer Fold 2/5

Inner Fold 1/5
Training until validation scores don't improve for 100 rounds
Early stopping, 

## 10. Results

The saved run reports:

| Outer fold | ROC-AUC |
|---|---:|
| 1 | 0.916370 |
| 2 | 0.917269 |
| 3 | 0.916751 |
| 4 | 0.917886 |
| 5 | 0.915014 |

**Overall out-of-fold ROC-AUC: 0.916630**; mean outer-fold ROC-AUC: 0.916658; standard deviation: 0.000967.

The notebook does not contain a verified Kaggle public or private leaderboard score or final competition rank. It does not establish causal effects on retention.


## 11. Export the Kaggle submission

Generate `submission.csv`, preserving sample-submission row IDs and using the averaged stacked model probabilities for `Churn`. The submitted CSV is not stored in this repository.


In [14]:
submission = pd.read_csv(r"/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv")
submission['Churn'] = pred_xgb
submission.to_csv("submission.csv", index=False)
submission.head()

,id,Churn
0,594194,0.062939
1,594195,0.000506
2,594196,0.104033
3,594197,0.003376
4,594198,0.460147


## Reproducibility and limitations

- Requires Kaggle competition files and the packages listed in `requirements.txt`; model training can be computationally intensive.
- Some saved training logs are retained for transparency. Scores reflect the supplied run, not a rerun here.
- Original public telco dataset was loaded but unused; no public dataset was merged into training.
- OOF target encoding and outer-fold-independent early stopping would strengthen evaluation.
- This is a predictive churn classification project, not a causal retention experiment or a production deployment.

See [README.md](README.md) for a step-by-step explanation and suggestions for improvement.
